# First look at the data: Comtrade

Goal: see what Comtrade actually returns before anything gets built on top of it. PortWatch has its own notebook, `02_portwatch_look.ipynb`.

Nothing here is production code. This is a scratchpad. Run cells top to bottom, read the output, and write what you notice in the last cell.

Kernel check: top right should say `.venv`. If it says anything else, switch it or the imports will fail.

In [2]:
import inspect
import os

import pandas as pd
import requests
from dotenv import load_dotenv

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# find .env whether the notebook runs from notebooks/ or the repo root
load_dotenv(dotenv_path="../.env")
load_dotenv()

key = os.environ.get("COMTRADE_API_KEY", "")
print(f"key present: {bool(key)}, length {len(key)}")

key present: True, length 32


---
## Part 1: Comtrade

Comtrade reports one row per reporter, partner, commodity code, year, and flow direction. For us:

- reporter = United States (country code `842`)
- partner = every country, one row each
- commodity = one HS6 code
- flow = `M` for imports

So a single call for one product in one year gives you roughly one row per supplier country. That shape is the entire basis of the concentration measure.

First, look at what the library offers rather than trusting anyone's memory of the argument names.

In [3]:
import comtradeapicall

funcs = [f for f in dir(comtradeapicall) if not f.startswith("_")]
print(funcs)

['Async', 'BulkDownload', 'DataAvailability', 'GroupCalc', 'Metadata', 'PreviewGet', 'SUV', 'bulkDownloadAndCombineFinalClassicFile', 'bulkDownloadAndCombineFinalFile', 'bulkDownloadAndCombineTarifflineFile', 'bulkDownloadFinalClassicFile', 'bulkDownloadFinalClassicFileDateRange', 'bulkDownloadFinalFile', 'bulkDownloadFinalFileDateRange', 'bulkDownloadTarifflineFile', 'bulkDownloadTarifflineFileDateRange', 'checkAsyncDataRequest', 'convertCountryIso3ToCode', 'downloadAsyncFinalDataRequest', 'downloadAsyncTarifflineDataRequest', 'getBilateralData', 'getCountFinalData', 'getFinalClassicDataBulkAvailability', 'getFinalData', 'getFinalDataAvailability', 'getFinalDataBulkAvailability', 'getLiveUpdate', 'getMetadata', 'getReference', 'getSUV', 'getTarifflineData', 'getTarifflineDataAvailability', 'getTarifflineDataBulkAvailability', 'getTradeBalance', 'getTradeMatrix', 'getTradeMatrixGroup', 'listReference', 'previewCountFinalData', 'previewFinalData', 'previewTarifflineData', 'submitAsyncFi

In [4]:
# the exact arguments previewFinalData expects, straight from the installed package
print(inspect.signature(comtradeapicall.previewFinalData))

(typeCode, freqCode, clCode, period, reporterCode, cmdCode, flowCode, partnerCode, partner2Code, customsCode, motCode, maxRecords=None, format_output=None, aggregateBy=None, breakdownMode=None, countOnly=None, includeDesc=None, proxy_url=None)


`previewFinalData` needs no API key and caps at 500 records. Perfect for looking around without spending calls from your daily budget.

`getFinalData` is the keyed version with the real limits. We use preview here.

If the call below errors on an unexpected keyword, compare against the signature printed above and adjust. The package changes argument names occasionally.

In [7]:
# 854232 = electronic integrated circuits, memory. One year, all partners.
df = comtradeapicall.previewFinalData(
    typeCode="C",        # C = commodities (as opposed to services)
    freqCode="A",        # A = annual
    clCode="HS",         # HS classification
    period="2023",
    reporterCode="842",  # United States
    cmdCode="854232",
    flowCode="M",        # M = imports
    partnerCode=None,    # None = all partners
    partner2Code=None,
    customsCode=None,
    motCode=None,
    includeDesc=True     # default for names is False 
)

print(type(df))
print(f"rows: {len(df)}")
df.head()

<class 'pandas.DataFrame'>
rows: 56


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,flowDesc,partnerCode,partnerISO,partnerDesc,partner2Code,partner2ISO,partner2Desc,classificationCode,classificationSearchCode,isOriginalClassification,cmdCode,cmdDesc,aggrLevel,isLeaf,customsCode,customsDesc,mosCode,motCode,motDesc,qtyUnitCode,qtyUnitAbbr,qty,isQtyEstimated,altQtyUnitCode,altQtyUnitAbbr,altQty,isAltQtyEstimated,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,842,USA,USA,M,Import,0,W00,World,0,W00,World,H6,HS,True,854232,Electronic integrated circuits; memories,6,True,C00,TOTAL CPC,0,0,TOTAL MOT,5,u,2.379172e+09,False,5,u,2.379172e+09,False,1086123.760,True,0.0,False,2.032402e+09,2.022569e+09,2.032402e+09,4,False,True
1,C,A,20230101,2023,52,2023,842,USA,USA,M,Import,36,AUS,Australia,0,W00,World,H6,HS,True,854232,Electronic integrated circuits; memories,6,True,C00,TOTAL CPC,0,0,TOTAL MOT,5,u,4.878000e+03,False,5,u,4.878000e+03,False,38.747,True,0.0,False,7.250500e+04,7.218500e+04,7.250500e+04,4,True,False
2,C,A,20230101,2023,52,2023,842,USA,USA,M,Import,40,AUT,Austria,0,W00,World,H6,HS,True,854232,Electronic integrated circuits; memories,6,True,C00,TOTAL CPC,0,0,TOTAL MOT,5,u,1.339830e+05,False,5,u,1.339830e+05,False,1040.956,True,0.0,False,1.947882e+06,1.936611e+06,1.947882e+06,4,True,False
3,C,A,20230101,2023,52,2023,842,USA,USA,M,Import,56,BEL,Belgium,0,W00,World,H6,HS,True,854232,Electronic integrated circuits; memories,6,True,C00,TOTAL CPC,0,0,TOTAL MOT,5,u,4.007900e+04,False,5,u,4.007900e+04,False,45.586,True,0.0,False,8.530200e+04,8.329700e+04,8.530200e+04,4,True,False
4,C,A,20230101,2023,52,2023,842,USA,USA,M,Import,76,BRA,Brazil,0,W00,World,H6,HS,True,854232,Electronic integrated circuits; memories,6,True,C00,TOTAL CPC,0,0,TOTAL MOT,5,u,7.280000e+02,False,5,u,7.280000e+02,False,6.058,True,0.0,False,1.133600e+04,1.063500e+04,1.133600e+04,4,True,False


In [8]:
# every column it returned. There are a lot, and most are useless to us.
list(df.columns)

['typeCode',
 'freqCode',
 'refPeriodId',
 'refYear',
 'refMonth',
 'period',
 'reporterCode',
 'reporterISO',
 'reporterDesc',
 'flowCode',
 'flowDesc',
 'partnerCode',
 'partnerISO',
 'partnerDesc',
 'partner2Code',
 'partner2ISO',
 'partner2Desc',
 'classificationCode',
 'classificationSearchCode',
 'isOriginalClassification',
 'cmdCode',
 'cmdDesc',
 'aggrLevel',
 'isLeaf',
 'customsCode',
 'customsDesc',
 'mosCode',
 'motCode',
 'motDesc',
 'qtyUnitCode',
 'qtyUnitAbbr',
 'qty',
 'isQtyEstimated',
 'altQtyUnitCode',
 'altQtyUnitAbbr',
 'altQty',
 'isAltQtyEstimated',
 'netWgt',
 'isNetWgtEstimated',
 'grossWgt',
 'isGrossWgtEstimated',
 'cifvalue',
 'fobvalue',
 'primaryValue',
 'legacyEstimationFlag',
 'isReported',
 'isAggregate']

In [9]:
# how empty is it? columns that are entirely null are columns we never land in Postgres.
null_share = df.isna().mean().sort_values(ascending=False)
null_share.head(30)

typeCode                0.0
altQty                  0.0
mosCode                 0.0
motCode                 0.0
motDesc                 0.0
qtyUnitCode             0.0
qtyUnitAbbr             0.0
qty                     0.0
isQtyEstimated          0.0
altQtyUnitCode          0.0
altQtyUnitAbbr          0.0
isAltQtyEstimated       0.0
customsCode             0.0
netWgt                  0.0
isNetWgtEstimated       0.0
grossWgt                0.0
isGrossWgtEstimated     0.0
cifvalue                0.0
fobvalue                0.0
primaryValue            0.0
legacyEstimationFlag    0.0
isReported              0.0
customsDesc             0.0
isLeaf                  0.0
freqCode                0.0
partnerCode             0.0
refPeriodId             0.0
refYear                 0.0
refMonth                0.0
period                  0.0
dtype: float64

### The columns that matter

Out of all that, the concentration measure needs about five: the partner country, the trade value, the year, the commodity code, and the flow direction. Everything else is either metadata or empty.

Watch for a row where the partner is `World`. Comtrade includes an aggregate row alongside the individual countries. If you leave it in, every share you compute is wrong by roughly half. This is the single most common mistake with this dataset.

In [12]:
cols = [c for c in ["period", "reporterDesc", "partnerDesc", "partnerCode", "cmdCode", "flowDesc", "primaryValue", "netWgt"] if c in df.columns]
sample = df[cols].sort_values("primaryValue", ascending=False)
sample.head(15)

,period,reporterDesc,partnerDesc,partnerCode,cmdCode,flowDesc,primaryValue,netWgt
0,2023,USA,World,0,854232,Import,2.032402e+09,1086123.760
34,2023,USA,"Other Asia, nes",490,854232,Import,7.285193e+08,389323.597
31,2023,USA,Malaysia,458,854232,Import,5.144736e+08,274936.718
28,2023,USA,Rep. of Korea,410,854232,Import,2.364938e+08,126383.236
44,2023,USA,Singapore,702,854232,Import,1.232630e+08,65872.209
26,2023,USA,Japan,392,854232,Import,1.177809e+08,62942.551
8,2023,USA,China,156,854232,Import,1.076433e+08,57524.992
52,2023,USA,Thailand,764,854232,Import,8.711463e+07,46554.399
38,2023,USA,Philippines,608,854232,Import,4.383585e+07,23426.050
16,2023,USA,France,251,854232,Import,2.746971e+07,14679.922


In [13]:
# is the World aggregate in here?
df[df["partnerDesc"].astype(str).str.contains("World", case=False, na=False)][cols]

,period,reporterDesc,partnerDesc,partnerCode,cmdCode,flowDesc,primaryValue,netWgt
0,2023,USA,World,0,854232,Import,2.032402e+09,1086123.76


### What concentration actually is

Five lines. Drop the World row, turn each country's value into a share of the total, square the shares, sum them, scale to 10,000.

One country supplying everything gives 10,000. A hundred equal suppliers gives 100. That number is the spine of the whole project.

This runs in pandas here because you are looking at it. In the real build it becomes a dbt model in SQL, because it has to run over every product and every year on a schedule.

In [9]:
d = df[~df["partnerDesc"].astype(str).str.contains("World", case=False, na=False)].copy()
d = d[d["primaryValue"] > 0]

d["share"] = d["primaryValue"] / d["primaryValue"].sum()
hhi = (d["share"] ** 2).sum() * 10_000

print(f"suppliers: {len(d)}")
print(f"HHI: {hhi:,.0f}")
print(f"top supplier share: {d['share'].max():.1%}")
print(f"top 3 share: {d.nlargest(3, 'share')['share'].sum():.1%}")

d.nlargest(10, "share")[["partnerDesc", "primaryValue", "share"]]

suppliers: 55
HHI: 2,186
top supplier share: 35.8%
top 3 share: 72.8%


,partnerDesc,primaryValue,share
34,"Other Asia, nes",728519335.0,0.358452
31,Malaysia,514473606.0,0.253136
28,Rep. of Korea,236493837.0,0.116362
44,Singapore,123262958.0,0.060649
26,Japan,117780853.0,0.057952
8,China,107643280.0,0.052964
52,Thailand,87114626.0,0.042863
38,Philippines,43835848.0,0.021568
16,France,27469712.0,0.013516
7,Canada,21766526.0,0.010710


### Now try a code from the other basket

`850511` is permanent magnets. Change the code below and rerun. Compare the HHI to the one you just got.

This is open question OQ-1 and the analytical note in the brief, answered by looking rather than assuming.

In [10]:
def quick_look(cmd_code, year="2023"):
    """One product, one year. Returns the frame and prints the headline numbers."""
    f = comtradeapicall.previewFinalData(
        typeCode="C", freqCode="A", clCode="HS", period=str(year),
        reporterCode="842", cmdCode=str(cmd_code), flowCode="M",
        partnerCode=None, partner2Code=None, customsCode=None, motCode=None, includeDesc=True
    )
    if f is None or len(f) == 0:
        print(f"{cmd_code} {year}: NO DATA")
        return None

    g = f[~f["partnerDesc"].astype(str).str.contains("World", case=False, na=False)].copy()
    g = g[g["primaryValue"] > 0]
    if len(g) == 0:
        print(f"{cmd_code} {year}: no positive values")
        return f

    g["share"] = g["primaryValue"] / g["primaryValue"].sum()
    top = g.nlargest(1, "share").iloc[0]
    print(
        f"{cmd_code} {year}: {len(g)} suppliers, "
        f"HHI {(g['share'] ** 2).sum() * 10_000:,.0f}, "
        f"total ${g['primaryValue'].sum() / 1e9:,.1f}B, "
        f"top {top['partnerDesc']} at {top['share']:.1%}"
    )
    return g


magnets = quick_look("850511")
magnets.nlargest(8, "share")[["partnerDesc", "primaryValue", "share"]] if magnets is not None else None

850511 2023: 55 suppliers, HHI 6,413, total $0.6B, top China at 79.8%


,partnerDesc,primaryValue,share
10,China,445485216.0,0.798063
25,Japan,28039279.0,0.050231
18,Germany,15949284.0,0.028572
32,"Other Asia, nes",7139515.0,0.012790
40,Philippines,7110328.0,0.012738
16,Finland,6465944.0,0.011583
21,Indonesia,5813089.0,0.010414
51,Switzerland,5671453.0,0.010160


In [11]:
# run the rest of the candidates from PROJECT_BRIEF section 3.
# codes that come back NO DATA, or with tiny totals, are candidates to cut from the baskets.
for code in ["854231", "854232", "854233", "854239", "854110", "854121", "848620", "280530", "284690", "850511"]:
    quick_look(code)

854231 2023: 93 suppliers, HHI 1,590, total $20.3B, top Malaysia at 32.8%


854232 2023: 55 suppliers, HHI 2,186, total $2.0B, top Other Asia, nes at 35.8%


854233 2023: 52 suppliers, HHI 1,658, total $0.8B, top Malaysia at 30.2%


854239 2023: 90 suppliers, HHI 1,573, total $12.7B, top Other Asia, nes at 28.8%


854110 2023: 53 suppliers, HHI 1,522, total $0.5B, top China at 31.1%


854121 2023: 38 suppliers, HHI 1,652, total $0.1B, top China at 29.4%


848620 2023: 46 suppliers, HHI 1,893, total $5.9B, top Japan at 36.2%


280530 2023: 12 suppliers, HHI 9,749, total $0.0B, top China at 98.7%


284690 2023: 23 suppliers, HHI 4,750, total $0.2B, top China at 67.2%


850511 2023: 55 suppliers, HHI 6,413, total $0.6B, top China at 79.8%


In [44]:
f = comtradeapicall.previewFinalData(
    typeCode="C", freqCode="A", clCode="HS", period="2025",
    reporterCode="842", cmdCode="854232", flowCode="M",
    partnerCode=None, partner2Code=None, customsCode=None,
    motCode=None, includeDesc=True,
)

print([c for c in f.columns if "mot" in c.lower()])
print(f[["motCode", "motDesc"]].drop_duplicates() if "motCode" in f.columns else "no mot columns returned")

['motCode', 'motDesc']
   motCode    motDesc
0        0  TOTAL MOT


---
## What you found

Write it here in plain English. Not for me, for you. This is what turns into the verification findings and eventually into how you talk about the project in an interview.

- Comtrade returns roughly ... rows for one product-year, one per ...
- The World row: present / absent, handled by ...
- HHI for integrated circuits: ... Top supplier: ...
- HHI for permanent magnets: ... Top supplier: ...
- Is the magnets concentration worse than the raw materials? ...
- Codes that returned no data: ...
- Does the US report a mode of transport breakdown: ...
- Things I do not understand yet: ...